# 🔗 데이터 통합 (전체 파이프라인)

## 목적
1. **Part 1**: KoBigBird OOF 통합 + 성능 평가
2. **Part 2**: 최종 데이터 병합 + 결측치 처리

## Part 1: KoBigBird OOF 통합
### 입력
- `oof_pred_AUDIT_fold1~4_seed42.csv` (4개)
- `oof_pred_ETC_fold1~4_seed42.csv` (4개)
- `oof_pred_MDA_fold1~4_seed42.csv` (4개)

### 출력
- `text_features_kobigbird_final.csv`
- **성능 지표**: 각 데이터셋별 AUC, Accuracy, Precision, Recall

## Part 2: 최종 데이터 병합
### 입력
1. `Bankruptcy_Dataset_For_Modeling_With_Macro.csv` (수치 데이터)
2. `text_features_kobigbird_final.csv` (Part 1 결과)
3. `text_features_lexicon_with_year.csv` (감성사전)

### 출력
- `최종_통합데이터_모델링용.csv`

### 결측치 처리 전략
- **KoBigBird 확률**: 0으로 채움 (텍스트 데이터가 없는 경우)
- **감성사전 피처**: 0으로 채움 (텍스트 데이터가 없는 경우)
- **수치 데이터**: 그대로 유지 (모델에서 처리)

---
# 📊 Part 1: KoBigBird OOF 통합 + 성능 평가
---

## 📦 Step 1: 라이브러리 임포트 및 설정

In [10]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✅ 라이브러리 로드 완료")
print(f"   - Pandas 버전: {pd.__version__}")
print(f"   - NumPy 버전: {np.__version__}")
print(f"   - Scikit-learn 임포트 완료 (성능 지표 계산용)")

✅ 라이브러리 로드 완료
   - Pandas 버전: 2.3.3
   - NumPy 버전: 2.4.0
   - Scikit-learn 임포트 완료 (성능 지표 계산용)


## ⚙️ Step 2: KoBigBird 경로 설정

In [13]:
# ============================================
# KoBigBird OOF 경로 설정
# ============================================

INPUT_DIR = "/Users/taerimeom/insight/잡았다요놈/Kobigbird"
OUTPUT_DIR = INPUT_DIR

SEED = 42
DATASETS = ["AUDIT", "ETC", "MDA"]
N_FOLDS = 4

print("="*80)
print("📂 KoBigBird 경로 설정")
print("="*80)
print(f"\n[입력 폴더]: {INPUT_DIR}")
print(f"[데이터셋]: {DATASETS}")
print(f"[Fold 수]: {N_FOLDS}개/데이터셋")
print("="*80)

📂 KoBigBird 경로 설정

[입력 폴더]: /Users/taerimeom/insight/잡았다요놈/Kobigbird
[데이터셋]: ['AUDIT', 'ETC', 'MDA']
[Fold 수]: 4개/데이터셋


## 📂 Step 3: 각 데이터셋 4-Fold 통합 + 성능 평가

In [20]:
# ============================================
# Step 3: 각 데이터셋의 4개 Fold 통합 + 성능 평가
# ============================================
print("\n" + "="*80)
print("🔗 Step 3: 각 데이터셋 4-Fold 통합 + 성능 평가")
print("="*80)
print()

dataset_oof_files = {}
dataset_metrics = {}

for dataset in DATASETS:
    print(f"\n{'─'*80}")
    print(f"📦 {dataset} 데이터셋")
    print(f"{'─'*80}")
    
    # Fold 파일 찾기
    fold_files = []
    for fold in range(1, N_FOLDS + 1):
        file_path = os.path.join(INPUT_DIR, f"oof_pred_{dataset}_fold{fold}_seed{SEED}.csv")
        if os.path.exists(file_path):
            fold_files.append(file_path)
            file_size = os.path.getsize(file_path) / 1024
            print(f"   ✅ Fold {fold}: {Path(file_path).name} ({file_size:.1f} KB)")
    
    if len(fold_files) != N_FOLDS:
        print(f"\n   ⚠️  경고: {len(fold_files)}/{N_FOLDS}개만 존재")
        continue
    
    # Fold 통합
    fold_dfs = [pd.read_csv(f) for f in fold_files]
    df_combined = pd.concat(fold_dfs, axis=0, ignore_index=True)
    df_combined = df_combined.sort_values(["corp_code", "target_year"]).reset_index(drop=True)
    
    print(f"\n   ✅ 통합 완료: {len(df_combined):,}개 샘플")
    
    # 📊 성능 평가
    y_true = df_combined["y"]
    y_pred_prob = df_combined["pred_prob"]
    y_pred = (y_pred_prob >= 0.5).astype(int)
    
    metrics = {
        "AUC": roc_auc_score(y_true, y_pred_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0)
    }
    
    dataset_metrics[dataset] = metrics
    
    print(f"\n   📊 성능 지표 (OOF):")
    for metric_name, metric_value in metrics.items():
        print(f"      {metric_name:12s}: {metric_value:.4f}")
    
    # 컬럼명 변경 및 저장
    df_output = df_combined[["corp_code", "target_year", "y", "pred_prob"]].copy()
    df_output = df_output.rename(columns={"pred_prob": f"{dataset.lower()}_prob"})
    
    output_file = os.path.join(OUTPUT_DIR, f"text_features_{dataset}_oof.csv")
    df_output.to_csv(output_file, index=False, encoding="utf-8-sig")
    dataset_oof_files[dataset] = output_file
    
    print(f"\n   💾 저장: {output_file}")

print("\n" + "="*80)
print("✅ Step 3 완료")
print("="*80)


🔗 Step 3: 각 데이터셋 4-Fold 통합 + 성능 평가


────────────────────────────────────────────────────────────────────────────────
📦 AUDIT 데이터셋
────────────────────────────────────────────────────────────────────────────────
   ✅ Fold 1: oof_pred_AUDIT_fold1_seed42.csv (197.5 KB)
   ✅ Fold 2: oof_pred_AUDIT_fold2_seed42.csv (194.4 KB)
   ✅ Fold 3: oof_pred_AUDIT_fold3_seed42.csv (195.4 KB)
   ✅ Fold 4: oof_pred_AUDIT_fold4_seed42.csv (181.3 KB)

   ✅ 통합 완료: 28,222개 샘플

   📊 성능 지표 (OOF):
      AUC         : 0.9182
      Accuracy    : 0.9047
      Precision   : 0.9054
      Recall      : 0.6202
      F1-Score    : 0.7361

   💾 저장: /Users/taerimeom/insight/잡았다요놈/Kobigbird/text_features_AUDIT_oof.csv

────────────────────────────────────────────────────────────────────────────────
📦 ETC 데이터셋
────────────────────────────────────────────────────────────────────────────────
   ✅ Fold 1: oof_pred_ETC_fold1_seed42.csv (186.3 KB)
   ✅ Fold 2: oof_pred_ETC_fold2_seed42.csv (190.4 KB)
   ✅ Fold 3: oof_pred_ET

## 📊 Step 3-1: 데이터셋별 성능 요약

In [21]:
# ============================================
# Step 3-1: 데이터셋별 성능 요약
# ============================================
print("\n" + "="*80)
print("📊 데이터셋별 KoBigBird 성능 요약")
print("="*80)
print()

# 성능 지표를 DataFrame으로 정리
metrics_df = pd.DataFrame(dataset_metrics).T
metrics_df.index.name = "Dataset"

print("📈 전체 성능 비교:")
display(metrics_df)

print(f"\n🏆 최고 성능:")
best_dataset = metrics_df["AUC"].idxmax()
best_auc = metrics_df["AUC"].max()
print(f"   - 데이터셋: {best_dataset}")
print(f"   - AUC: {best_auc:.4f}")

print(f"\n📊 평균 성능:")
for metric in metrics_df.columns:
    avg_val = metrics_df[metric].mean()
    print(f"   - {metric}: {avg_val:.4f}")


📊 데이터셋별 KoBigBird 성능 요약

📈 전체 성능 비교:


,AUC,Accuracy,Precision,Recall,F1-Score
Dataset,,,,,
AUDIT,0.918248,0.904720,0.905383,0.620205,0.736140
ETC,0.659230,0.794699,0.576139,0.158896,0.249093
MDA,0.699695,0.789313,0.518425,0.237269,0.325544



🏆 최고 성능:
   - 데이터셋: AUDIT
   - AUC: 0.9182

📊 평균 성능:
   - AUC: 0.7591
   - Accuracy: 0.8296
   - Precision: 0.6666
   - Recall: 0.3388
   - F1-Score: 0.4369


## 🔗 Step 4: 3개 데이터셋 병합

In [25]:
# ============================================
# Step 4: 3개 데이터셋 병합
# ============================================
print("\n" + "="*80)
print("🔗 Step 4: 3개 데이터셋 병합")
print("="*80)
print()

# 로드 및 중복 제거
dfs = {}
for dataset in DATASETS:
    file_path = os.path.join(OUTPUT_DIR, f"text_features_{dataset}_oof.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        
        # 중복 제거 (corp_code, target_year 기준)
        before = len(df)
        df = df.drop_duplicates(subset=['corp_code', 'target_year'])
        after = len(df)
        
        dfs[dataset] = df
        
        if before != after:
            print(f"✅ {dataset}: {after:,}개 (중복 {before-after:,}개 제거)")
        else:
            print(f"✅ {dataset}: {len(df):,}개")

# 병합 (inner join)
df_kobigbird = dfs["AUDIT"][["corp_code", "target_year", "y", "audit_prob"]].copy()
print(f"\n기준: AUDIT ({len(df_kobigbird):,}개)")

df_kobigbird = df_kobigbird.merge(
    dfs["ETC"][["corp_code", "target_year", "etc_prob"]],
    on=["corp_code", "target_year"],
    how="inner"
)
print(f"+ ETC: {len(df_kobigbird):,}개")

df_kobigbird = df_kobigbird.merge(
    dfs["MDA"][["corp_code", "target_year", "mda_prob"]],
    on=["corp_code", "target_year"],
    how="inner"
)
print(f"+ MDA: {len(df_kobigbird):,}개")

# 결측치 처리
print(f"\n🔧 결측치 처리:")
prob_cols = ["audit_prob", "etc_prob", "mda_prob"]
for col in prob_cols:
    n_missing = df_kobigbird[col].isnull().sum()
    if n_missing > 0:
        print(f"   {col}: {n_missing:,}개 → 0")
        df_kobigbird[col] = df_kobigbird[col].fillna(0)
    else:
        print(f"   {col}: 결측 없음 ✅")

df_kobigbird = df_kobigbird.sort_values(["corp_code", "target_year"]).reset_index(drop=True)

print(f"\n✅ 병합 완료: {len(df_kobigbird):,}행 × {len(df_kobigbird.columns)}열")



🔗 Step 4: 3개 데이터셋 병합

✅ AUDIT: 24,617개 (중복 3,605개 제거)
✅ ETC: 24,617개 (중복 3,605개 제거)
✅ MDA: 24,617개 (중복 3,605개 제거)

기준: AUDIT (24,617개)
+ ETC: 24,617개
+ MDA: 24,617개

🔧 결측치 처리:
   audit_prob: 결측 없음 ✅
   etc_prob: 결측 없음 ✅
   mda_prob: 결측 없음 ✅

✅ 병합 완료: 24,617행 × 6열


## 💾 Step 5: KoBigBird 최종 파일 저장

In [26]:
# ============================================
# Step 5: KoBigBird 최종 파일 저장
# ============================================
print("\n" + "="*80)
print("💾 Step 5: KoBigBird 최종 파일 저장")
print("="*80)
print()

KOBIGBIRD_OUTPUT = "text_features_kobigbird_final.csv"
df_kobigbird.to_csv(KOBIGBIRD_OUTPUT, index=False, encoding="utf-8-sig")

file_size = os.path.getsize(KOBIGBIRD_OUTPUT) / 1024 / 1024

print(f"✅ 저장 완료!")
print(f"   - 파일: {KOBIGBIRD_OUTPUT}")
print(f"   - 크기: {file_size:.2f} MB")
print(f"   - 샘플: {len(df_kobigbird):,}개")
print(f"   - 컬럼: {list(df_kobigbird.columns)}")

print(f"\n📋 샘플 데이터:")
display(df_kobigbird.head())

print(f"\n✅ Part 1 완료: KoBigBird OOF 통합 + 성능 평가")


💾 Step 5: KoBigBird 최종 파일 저장

✅ 저장 완료!
   - 파일: text_features_kobigbird_final.csv
   - 크기: 1.12 MB
   - 샘플: 24,617개
   - 컬럼: ['corp_code', 'target_year', 'y', 'audit_prob', 'etc_prob', 'mda_prob']

📋 샘플 데이터:


,corp_code,target_year,y,audit_prob,etc_prob,mda_prob
0,100258,2015,1,0.995066,0.147678,0.611208
1,100258,2016,1,0.995018,0.498459,0.503921
2,100258,2017,1,0.988863,0.496212,0.500870
3,100258,2018,1,0.913984,0.630486,0.329068
4,100258,2019,1,0.667744,0.582432,0.349644



✅ Part 1 완료: KoBigBird OOF 통합 + 성능 평가


------

In [42]:
# 최종통합.ipynb

# ============================================
# 📊 최종 데이터 통합
# ============================================
# 
# 목적: 텍스트 피처 + 수치 데이터 최종 병합
# 
# 입력:

# 1. text_features_with_stock_code_final.csv (텍스트 피처)
#    - KoBigBird 확률 (audit_prob, etc_prob, mda_prob)
#    - 감성사전 피처 (lex_*)
#    - 종목 코드 (stock_code, company_name)
# 
# 2. Bankruptcy_Dataset_For_Modeling_With_Macro.csv (수치 데이터)
#    - 재무 피처 (F1_*, F2_*, F3_*, F4_*)
#    - 거시경제 피처 (M_*)
# 
# 출력:
# - 최종_통합데이터_모델링용.csv
# 
# 병합 키: (stock_code, target_year)
# ============================================


## 📦 Step 1: 라이브러리 임포트 및 설정


In [43]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print("✅ 라이브러리 로드 완료")
print(f"   - Pandas 버전: {pd.__version__}")
print(f"   - NumPy 버전: {np.__version__}")


✅ 라이브러리 로드 완료
   - Pandas 버전: 2.3.3
   - NumPy 버전: 2.4.0


In [44]:
# ============================================
# Step 2: 텍스트 피처 로드
# ============================================
print("\n" + "="*80)
print("📂 Step 2: 텍스트 피처 로드")
print("="*80)
print()

TEXT_FEATURES_FILE = "text_features_with_stock_code_final.csv"

if not Path(TEXT_FEATURES_FILE).exists():
    print(f"❌ 파일 없음: {TEXT_FEATURES_FILE}")
    raise FileNotFoundError(TEXT_FEATURES_FILE)

df_text = pd.read_csv(TEXT_FEATURES_FILE)

print(f"✅ 텍스트 피처 로드 완료")
print(f"   - 행: {len(df_text):,}개")
print(f"   - 열: {len(df_text.columns)}개")
print(f"   - 기업 수: {df_text['stock_code'].nunique():,}개")
print(f"   - 연도 범위: {df_text['target_year'].min()} ~ {df_text['target_year'].max()}")

print(f"\n📋 컬럼 목록:")
for i, col in enumerate(df_text.columns, 1):
    print(f"   {i:2d}. {col}")

print(f"\n📋 샘플 데이터:")
display(df_text.head(3))

# 필수 컬럼 확인
required_cols = ['stock_code', 'target_year', 'company_name']
missing_cols = [col for col in required_cols if col not in df_text.columns]
if missing_cols:
    print(f"\n❌ 필수 컬럼 누락: {missing_cols}")
    raise ValueError(f"필수 컬럼이 없습니다: {missing_cols}")

print(f"\n✅ 필수 컬럼 확인 완료")



📂 Step 2: 텍스트 피처 로드

✅ 텍스트 피처 로드 완료
   - 행: 24,617개
   - 열: 8개
   - 기업 수: 2,976개
   - 연도 범위: 2015 ~ 2025

📋 컬럼 목록:
    1. corp_code
    2. target_year
    3. y
    4. audit_prob
    5. etc_prob
    6. mda_prob
    7. stock_code
    8. company_name

📋 샘플 데이터:


,corp_code,target_year,y,audit_prob,etc_prob,mda_prob,stock_code,company_name
0,100258,2015,1,0.995066,0.147678,0.611208,030270,에스마크
1,100258,2016,1,0.995018,0.498459,0.503921,030270,에스마크
2,100258,2017,1,0.988863,0.496212,0.500870,030270,에스마크



✅ 필수 컬럼 확인 완료


In [54]:
# ============================================
# Step 2-1: 감성사전 로드 및 평균 계산
# ============================================
print("\n" + "="*80)
print("📂 Step 2-1: 감성사전 로드 및 평균 계산")
print("="*80)
print()

LEXICON_FILE = "text_features_lexicon.csv"

if not Path(LEXICON_FILE).exists():
    print(f"❌ 파일 없음: {LEXICON_FILE}")
    raise FileNotFoundError(LEXICON_FILE)

df_lexicon_raw = pd.read_csv(LEXICON_FILE)

print(f"✅ 감성사전 로드 완료")
print(f"   - 원본 행: {len(df_lexicon_raw):,}개")
print(f"   - 열: {len(df_lexicon_raw.columns)}개")

# 감성 피처 목록
lex_feature_cols = [c for c in df_lexicon_raw.columns if c.startswith('lex_')]
print(f"   - 감성 피처: {len(lex_feature_cols)}개")
print(f"     {lex_feature_cols}")

# 중복 확인
unique_keys = df_lexicon_raw.groupby(['corp_code', 'target_year']).ngroups
print(f"\n📊 데이터 구조:")
print(f"   - 고유 (corp_code, target_year): {unique_keys:,}개")
print(f"   - 평균 행/키: {len(df_lexicon_raw) / unique_keys:.1f}개")
print(f"   → AUDIT, MDA, ETC 각각 처리된 것으로 추정")

# 평균 계산
print(f"\n🔧 같은 (corp_code, target_year)의 감성 피처 평균 계산 중...")

# 평균 계산할 피처들
agg_dict = {col: 'mean' for col in lex_feature_cols}

df_lexicon = df_lexicon_raw.groupby(['corp_code', 'target_year']).agg(agg_dict).reset_index()

print(f"\n✅ 평균 계산 완료")
print(f"   - 결과 행: {len(df_lexicon):,}개")
print(f"   - 감소: {len(df_lexicon_raw):,} → {len(df_lexicon):,} ({len(df_lexicon_raw) - len(df_lexicon):,}개 감소)")

print(f"\n📋 평균 계산 후 샘플:")
display(df_lexicon.head(3))

# 통계 확인
print(f"\n📊 감성 피처 통계:")
display(df_lexicon[lex_feature_cols].describe())



📂 Step 2-1: 감성사전 로드 및 평균 계산

✅ 감성사전 로드 완료
   - 원본 행: 83,863개
   - 열: 11개
   - 감성 피처: 8개
     ['lex_sent_mean', 'lex_sent_sum', 'lex_pos_tf', 'lex_neg_tf', 'lex_pos_cnt', 'lex_neg_cnt', 'lex_abs_mean', 'lex_covered_tf']

📊 데이터 구조:
   - 고유 (corp_code, target_year): 24,617개
   - 평균 행/키: 3.4개
   → AUDIT, MDA, ETC 각각 처리된 것으로 추정

🔧 같은 (corp_code, target_year)의 감성 피처 평균 계산 중...

✅ 평균 계산 완료
   - 결과 행: 24,617개
   - 감소: 83,863 → 24,617 (59,246개 감소)

📋 평균 계산 후 샘플:


,corp_code,target_year,lex_sent_mean,lex_sent_sum,lex_pos_tf,lex_neg_tf,lex_pos_cnt,lex_neg_cnt,lex_abs_mean,lex_covered_tf
0,100258,2015,1.173744,56.407984,101.666667,73.333333,36.333333,30.333333,6.478039,175.000000
1,100258,2016,-1.875076,-644.980177,138.333333,178.000000,63.333333,68.000000,7.089901,316.333333
2,100258,2017,-1.759436,-752.436045,146.000000,207.333333,66.666667,78.333333,6.882544,353.333333



📊 감성 피처 통계:


,lex_sent_mean,lex_sent_sum,lex_pos_tf,lex_neg_tf,lex_pos_cnt,lex_neg_cnt,lex_abs_mean,lex_covered_tf
count,24617.000000,24617.000000,24617.000000,24617.000000,24617.000000,24617.000000,24617.000000,24617.000000
mean,0.255353,194.818926,150.712379,103.111122,61.847435,33.939544,6.230709,253.823501
std,1.213052,436.518973,85.979904,63.920675,27.614277,17.173540,0.748726,141.388204
min,-7.543090,-3975.688465,1.000000,1.666667,1.000000,1.000000,2.679812,2.666667
25%,-0.457949,-8.086897,88.666667,59.000000,44.000000,22.666667,6.088067,151.333333
50%,0.406140,188.031191,137.666667,89.666667,61.000000,31.333333,6.321521,232.333333
75%,1.102910,439.118301,205.333333,135.666667,80.333333,43.666667,6.550970,341.666667
max,3.212380,1855.757076,508.333333,446.666667,153.666667,129.666667,10.399735,761.666667


In [55]:
# ============================================
# Step 2-2: 텍스트 피처 병합
# ============================================
print("\n" + "="*80)
print("🔗 Step 2-2: 텍스트 피처 병합 (KoBigBird + 감성사전)")
print("="*80)
print()

# 베이스: KoBigBird (stock_code 포함)
df_text_combined = df_text.copy()
print(f"베이스 (KoBigBird): {len(df_text_combined):,}행")

# corp_code 표준화 (8자리 패딩)
df_text_combined['corp_code'] = df_text_combined['corp_code'].astype(str).str.zfill(8)
df_lexicon['corp_code'] = df_lexicon['corp_code'].astype(str).str.zfill(8)

# target_year 표준화
df_text_combined['target_year'] = df_text_combined['target_year'].astype(int)
df_lexicon['target_year'] = df_lexicon['target_year'].astype(int)

print(f"감성사전 (평균): {len(df_lexicon):,}행")

# 중복 제거 (혹시 모를 중복 체크)
df_lexicon_clean = df_lexicon.drop_duplicates(subset=['corp_code', 'target_year'])
if len(df_lexicon_clean) != len(df_lexicon):
    print(f"⚠️  감성사전 중복 제거: {len(df_lexicon)} → {len(df_lexicon_clean)}")

# LEFT JOIN으로 병합 (corp_code + target_year 기준)
df_text_combined = df_text_combined.merge(
    df_lexicon_clean,
    on=['corp_code', 'target_year'],
    how='left',
    suffixes=('', '_lex')
)

# y 컬럼 중복 처리 (KoBigBird의 y 유지)
if 'y_lex' in df_text_combined.columns:
    df_text_combined = df_text_combined.drop(columns=['y_lex'])

# 매칭 결과
lex_feature_cols = [c for c in df_lexicon_clean.columns if c.startswith('lex_')]
if len(lex_feature_cols) > 0:
    matched = df_text_combined[lex_feature_cols[0]].notna().sum()
    print(f"+ 감성사전: {matched:,}/{len(df_text_combined):,} ({matched/len(df_text_combined)*100:.1f}%) 매칭")

# 감성사전 결측치 0으로 채우기
print(f"\n🔧 감성사전 결측치 처리 (0으로 채우기):")
for col in lex_feature_cols:
    n_missing = df_text_combined[col].isnull().sum()
    if n_missing > 0:
        print(f"   {col}: {n_missing:,}개 → 0")
        df_text_combined[col] = df_text_combined[col].fillna(0)
    else:
        print(f"   {col}: 결측 없음 ✅")

print(f"\n✅ 텍스트 피처 병합 완료: {len(df_text_combined):,}행 × {len(df_text_combined.columns)}열")

# df_text를 업데이트 (이후 Step에서 사용)
df_text = df_text_combined.copy()

print(f"\n📋 최종 텍스트 피처 컬럼 ({len(df_text.columns)}개):")
for i, col in enumerate(df_text.columns, 1):
    print(f"   {i:2d}. {col}")

print(f"\n📋 샘플 데이터:")
display(df_text.head(3))



🔗 Step 2-2: 텍스트 피처 병합 (KoBigBird + 감성사전)

베이스 (KoBigBird): 24,617행
감성사전 (평균): 24,617행
+ 감성사전: 24,617/24,617 (100.0%) 매칭

🔧 감성사전 결측치 처리 (0으로 채우기):
   lex_sent_mean: 결측 없음 ✅
   lex_sent_sum: 결측 없음 ✅
   lex_pos_tf: 결측 없음 ✅
   lex_neg_tf: 결측 없음 ✅
   lex_pos_cnt: 결측 없음 ✅
   lex_neg_cnt: 결측 없음 ✅
   lex_abs_mean: 결측 없음 ✅
   lex_covered_tf: 결측 없음 ✅

✅ 텍스트 피처 병합 완료: 24,617행 × 16열

📋 최종 텍스트 피처 컬럼 (16개):
    1. corp_code
    2. target_year
    3. y
    4. audit_prob
    5. etc_prob
    6. mda_prob
    7. stock_code
    8. company_name
    9. lex_sent_mean
   10. lex_sent_sum
   11. lex_pos_tf
   12. lex_neg_tf
   13. lex_pos_cnt
   14. lex_neg_cnt
   15. lex_abs_mean
   16. lex_covered_tf

📋 샘플 데이터:


,corp_code,target_year,y,audit_prob,etc_prob,mda_prob,stock_code,company_name,lex_sent_mean,lex_sent_sum,lex_pos_tf,lex_neg_tf,lex_pos_cnt,lex_neg_cnt,lex_abs_mean,lex_covered_tf
0,00100258,2015,1,0.995066,0.147678,0.611208,00030270,에스마크,1.173744,56.407984,101.666667,73.333333,36.333333,30.333333,6.478039,175.000000
1,00100258,2016,1,0.995018,0.498459,0.503921,00030270,에스마크,-1.875076,-644.980177,138.333333,178.000000,63.333333,68.000000,7.089901,316.333333
2,00100258,2017,1,0.988863,0.496212,0.500870,00030270,에스마크,-1.759436,-752.436045,146.000000,207.333333,66.666667,78.333333,6.882544,353.333333


In [56]:
# ============================================
# Step 3: 수치 데이터 로드
# ============================================
print("\n" + "="*80)
print("📂 Step 3: 수치 데이터 로드")
print("="*80)
print()

NUMERIC_FILE = "Bankruptcy_Dataset_For_Modeling_With_Macro.csv"

if not Path(NUMERIC_FILE).exists():
    print(f"❌ 파일 없음: {NUMERIC_FILE}")
    raise FileNotFoundError(NUMERIC_FILE)

df_numeric = pd.read_csv(NUMERIC_FILE)

print(f"✅ 수치 데이터 로드 완료")
print(f"   - 행: {len(df_numeric):,}개")
print(f"   - 열: {len(df_numeric.columns)}개")
print(f"   - 메모리: {df_numeric.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n📋 샘플:")
display(df_numeric.head(3))



📂 Step 3: 수치 데이터 로드

✅ 수치 데이터 로드 완료
   - 행: 22,544개
   - 열: 22개
   - 메모리: 7.24 MB

📋 샘플:


,bsns_year,Company_Code,Company_Name,Status,F1_Equity_Growth,F1_Retained_Earnings_Ratio,F1_ROA,F1_Debt_Ratio,F1_Current_Ratio,F1_ROE,F1_Interest_Coverage,F2_KMV_DD,F3_Z_Score,F4_M_Score,M_Short_Term_Rate,M_Long_Term_Rate,M_Rate_Spread,M_Nominal_GDP_Growth,M_Real_GDP_Growth,M_Inflation,M_Exchange_Rate,Target
0,2015,071200,인피니트헬스케어,Distressed,NaN,0.251549,0.068161,0.251739,2.993100,0.091093,NaN,2.181834,3.509617,NaN,1.50,2.31,0.81,5.4,2.8,0.7,1131,1
1,2016,071200,인피니트헬스케어,Distressed,0.123921,0.308238,0.079259,0.231937,3.316511,0.103193,NaN,4.182605,3.183807,-1.008438,1.25,1.75,0.50,4.8,2.9,1.0,1160,1
2,2017,071200,인피니트헬스케어,Distressed,0.015111,0.326141,0.016567,0.038772,0.277537,0.021194,NaN,2.978670,2.767874,-1.150158,1.50,2.28,0.78,5.5,3.2,1.9,1131,1


In [57]:
# ============================================
# Step 4: 추이 피처 생성
# ============================================
print("\n" + "="*80)
print("📈 Step 4: 추이 피처 생성")
print("="*80)
print()

# 1. 추이 피처 대상 정의
trend_features = [
    'F1_Equity_Growth',
    'F1_Retained_Earnings_Ratio',
    'F1_ROA',
    'F1_Debt_Ratio',
    'F1_Current_Ratio',
    'F1_ROE',
    'F1_Interest_Coverage'
]

print(f"📊 추이 피처 대상: {len(trend_features)}개")
for i, feat in enumerate(trend_features, 1):
    if feat in df_numeric.columns:
        print(f"   {i}. {feat} ✅")
    else:
        print(f"   {i}. {feat} ❌ (컬럼 없음)")
print()

# 2. 정렬 (Company_Code, bsns_year 기준)
df_numeric_sorted = df_numeric.sort_values(['Company_Code', 'bsns_year']).reset_index(drop=True)
print(f"✅ 정렬 완료: Company_Code, bsns_year 기준")
print()

# 3. 전년도 데이터 생성
prev_year_cols = {}
for feature in trend_features:
    if feature in df_numeric_sorted.columns:
        df_numeric_sorted[f'{feature}_prev'] = df_numeric_sorted.groupby('Company_Code')[feature].shift(1)
        prev_year_cols[feature] = f'{feature}_prev'

print(f"✅ 전년도 데이터 생성: {len(prev_year_cols)}개 피처")
print()

# 4. 추이 피처 계산
trend_features_created = []

for feature in trend_features:
    if feature not in df_numeric_sorted.columns:
        continue

    prev_col = f'{feature}_prev'

    # (1) 절대 변화량
    change_col = f'{feature}_change'
    df_numeric_sorted[change_col] = df_numeric_sorted[feature] - df_numeric_sorted[prev_col]
    trend_features_created.append(change_col)

    # (2) 비율 변화 (%)
    pct_change_col = f'{feature}_pct_change'
    df_numeric_sorted[pct_change_col] = np.where(
        df_numeric_sorted[prev_col] != 0,
        (df_numeric_sorted[feature] - df_numeric_sorted[prev_col]) / df_numeric_sorted[prev_col] * 100,
        np.nan
    )
    trend_features_created.append(pct_change_col)

    # (3) 개선/악화 방향 (1=개선, 0=악화)
    direction_col = f'{feature}_improving'

    good_if_high = [
        'F1_Equity_Growth', 'F1_Retained_Earnings_Ratio', 'F1_ROA',
        'F1_Current_Ratio', 'F1_ROE', 'F1_Interest_Coverage'
    ]
    good_if_low = ['F1_Debt_Ratio']

    if feature in good_if_high:
        df_numeric_sorted[direction_col] = (df_numeric_sorted[feature] > df_numeric_sorted[prev_col]).astype(float)
    elif feature in good_if_low:
        df_numeric_sorted[direction_col] = (df_numeric_sorted[feature] < df_numeric_sorted[prev_col]).astype(float)

    trend_features_created.append(direction_col)

print(f"✅ 추이 피처 생성 완료: {len(trend_features_created)}개")
print(f"   - 절대 변화량: {len([c for c in trend_features_created if '_change' in c and '_pct_change' not in c])}개")
print(f"   - 비율 변화: {len([c for c in trend_features_created if '_pct_change' in c])}개")
print(f"   - 방향 지표: {len([c for c in trend_features_created if '_improving' in c])}개")
print()

# 5. 결측치 처리
for col in trend_features_created:
    n_missing_before = df_numeric_sorted[col].isnull().sum()
    if n_missing_before > 0:
        df_numeric_sorted[col] = df_numeric_sorted[col].fillna(0)

# 6. 이상값 처리 (비율 변화 ±1000% 제한)
pct_change_cols = [c for c in trend_features_created if '_pct_change' in c]
for col in pct_change_cols:
    n_clipped = ((df_numeric_sorted[col] < -1000) | (df_numeric_sorted[col] > 1000)).sum()
    if n_clipped > 0:
        df_numeric_sorted[col] = df_numeric_sorted[col].clip(-1000, 1000)

# 7. 전년도 컬럼 제거
df_numeric_sorted = df_numeric_sorted.drop(columns=list(prev_year_cols.values()))
print(f"🧹 전년도 임시 컬럼 제거: {len(prev_year_cols)}개")

# 8. df_numeric 업데이트
df_numeric = df_numeric_sorted.copy()
print(f"\n✅ 추이 피처 추가 완료!")
print(f"   - 총 샘플: {len(df_numeric):,}개")
print(f"   - 총 컬럼: {len(df_numeric.columns)}개")
print(f"   - 추가된 피처: {len(trend_features_created)}개")



📈 Step 4: 추이 피처 생성

📊 추이 피처 대상: 7개
   1. F1_Equity_Growth ✅
   2. F1_Retained_Earnings_Ratio ✅
   3. F1_ROA ✅
   4. F1_Debt_Ratio ✅
   5. F1_Current_Ratio ✅
   6. F1_ROE ✅
   7. F1_Interest_Coverage ✅

✅ 정렬 완료: Company_Code, bsns_year 기준

✅ 전년도 데이터 생성: 7개 피처

✅ 추이 피처 생성 완료: 21개
   - 절대 변화량: 7개
   - 비율 변화: 7개
   - 방향 지표: 7개

🧹 전년도 임시 컬럼 제거: 7개

✅ 추이 피처 추가 완료!
   - 총 샘플: 22,544개
   - 총 컬럼: 43개
   - 추가된 피처: 21개


In [58]:
# ============================================
# Step 5: 키 컬럼 표준화
# ============================================
print("\n" + "="*80)
print("🔧 Step 5: 키 컬럼 표준화")
print("="*80)
print()

# 텍스트 피처 표준화
df_text['stock_code'] = df_text['stock_code'].astype(str).str.zfill(8)
df_text['target_year'] = df_text['target_year'].astype(int)
print("✅ 텍스트: stock_code 8자리 패딩, target_year 정수 변환")

# 수치 데이터 표준화
df_numeric['Company_Code'] = df_numeric['Company_Code'].astype(str).str.zfill(8)
df_numeric['target_year'] = df_numeric['bsns_year'].astype(int)
print("✅ 수치: Company_Code 8자리 패딩, target_year 생성")

print("\n✅ 모든 키 컬럼 표준화 완료")
print(f"   - 텍스트: stock_code={df_text['stock_code'].iloc[0]}, target_year={df_text['target_year'].iloc[0]}")
print(f"   - 수치: Company_Code={df_numeric['Company_Code'].iloc[0]}, target_year={df_numeric['target_year'].iloc[0]}")



🔧 Step 5: 키 컬럼 표준화

✅ 텍스트: stock_code 8자리 패딩, target_year 정수 변환
✅ 수치: Company_Code 8자리 패딩, target_year 생성

✅ 모든 키 컬럼 표준화 완료
   - 텍스트: stock_code=00030270, target_year=2015
   - 수치: Company_Code=00000010, target_year=2023


In [59]:
# ============================================
# Step 6: 최종 병합
# ============================================
print("\n" + "="*80)
print("🔗 Step 6: 최종 병합 (stock_code + target_year)")
print("="*80)
print()

# 베이스: 수치 데이터
df_final = df_numeric.copy()
df_final = df_final.rename(columns={'Company_Code': 'stock_code'})
initial_rows = len(df_final)
print(f"베이스 (수치): {initial_rows:,}행")

# 중복 제거
df_text_clean = df_text.drop_duplicates(subset=['stock_code', 'target_year'])
print(f"텍스트 (중복 제거): {len(df_text_clean):,}행")

# 병합할 컬럼 선택 (y 제외)
text_feature_cols = [c for c in df_text_clean.columns 
                     if c not in ['stock_code', 'target_year', 'y']]
merge_cols = ['stock_code', 'target_year'] + text_feature_cols

# LEFT JOIN
df_final = pd.merge(
    df_final,
    df_text_clean[merge_cols],
    on=['stock_code', 'target_year'],
    how='left'
)

# 매칭 결과
matched = df_final[text_feature_cols[0]].notna().sum() if len(text_feature_cols) > 0 else 0
print(f"+ 텍스트 피처: {matched:,}/{initial_rows:,} ({matched/initial_rows*100:.1f}%) 매칭")

print(f"\n✅ 병합 완료: {len(df_final):,}행 × {len(df_final.columns)}열")



🔗 Step 6: 최종 병합 (stock_code + target_year)

베이스 (수치): 22,544행
텍스트 (중복 제거): 24,617행
+ 텍스트 피처: 22,496/22,544 (99.8%) 매칭

✅ 병합 완료: 22,544행 × 57열


In [60]:
# ============================================
# Step 7: 결측치 처리
# ============================================
print("\n" + "="*80)
print("🔧 Step 7: 결측치 처리")
print("="*80)
print()

# 텍스트 피처 결측치 처리 (0으로 채우기)
print("🔄 텍스트 피처 결측치 처리 (0으로 채우기):")

# KoBigBird 확률
kb_cols = ['audit_prob', 'etc_prob', 'mda_prob']
for col in kb_cols:
    if col in df_final.columns:
        n_missing = df_final[col].isnull().sum()
        if n_missing > 0:
            print(f"   {col}: {n_missing:,}개 → 0")
            df_final[col] = df_final[col].fillna(0)

# 감성사전 피처
lex_cols = [c for c in df_final.columns if c.startswith('lex_')]
if len(lex_cols) > 0:
    print(f"\n   감성사전 피처 ({len(lex_cols)}개):")
    for col in lex_cols:
        n_missing = df_final[col].isnull().sum()
        if n_missing > 0:
            print(f"      {col}: {n_missing:,}개 → 0")
            df_final[col] = df_final[col].fillna(0)

print(f"\n📊 수치형 피처: 결측치 그대로 유지 (모델에서 처리)")

print(f"\n✅ 결측치 처리 완료")



🔧 Step 7: 결측치 처리

🔄 텍스트 피처 결측치 처리 (0으로 채우기):
   audit_prob: 48개 → 0
   etc_prob: 48개 → 0
   mda_prob: 48개 → 0

   감성사전 피처 (8개):
      lex_sent_mean: 48개 → 0
      lex_sent_sum: 48개 → 0
      lex_pos_tf: 48개 → 0
      lex_neg_tf: 48개 → 0
      lex_pos_cnt: 48개 → 0
      lex_neg_cnt: 48개 → 0
      lex_abs_mean: 48개 → 0
      lex_covered_tf: 48개 → 0

📊 수치형 피처: 결측치 그대로 유지 (모델에서 처리)

✅ 결측치 처리 완료


In [61]:
# ============================================
# Step 8: 데이터 정리
# ============================================
print("\n" + "="*80)
print("🧹 Step 8: 데이터 정리")
print("="*80)
print()

# 중복 제거
before = len(df_final)
df_final = df_final.drop_duplicates(subset=['stock_code', 'target_year'])
after = len(df_final)

if before != after:
    print(f"⚠️  중복 제거: {before:,} → {after:,} ({before-after}개)")
else:
    print(f"✅ 중복 없음")

# 정렬
df_final = df_final.sort_values(['stock_code', 'target_year']).reset_index(drop=True)

print(f"\n✅ 정리 완료")
print(f"   - 샘플: {len(df_final):,}개")
print(f"   - 컬럼: {len(df_final.columns)}개")
print(f"   - 기업 수: {df_final['stock_code'].nunique():,}개")
print(f"   - 연도: {df_final['target_year'].min()} ~ {df_final['target_year'].max()}")



🧹 Step 8: 데이터 정리

⚠️  중복 제거: 22,544 → 19,603 (2941개)

✅ 정리 완료
   - 샘플: 19,603개
   - 컬럼: 57개
   - 기업 수: 2,803개
   - 연도: 2015 ~ 2025


In [62]:
# ============================================
# Step 9: 최종 검증
# ============================================
print("\n" + "="*80)
print("📊 Step 9: 최종 검증")
print("="*80)
print()

# 타겟 분포
if 'Target' in df_final.columns:
    print("1️⃣ 타겟 분포:")
    for label, count in df_final['Target'].value_counts().sort_index().items():
        label_name = "부도" if label == 1 else "정상"
        pct = count / len(df_final) * 100
        print(f"   {label_name} (Target={label}): {count:,}개 ({pct:.2f}%)")

# 샘플 확인
print(f"\n2️⃣ 데이터 샘플:")
display(df_final.head())

# 결측치 요약
print(f"\n3️⃣ 결측치 요약 (상위 10개):")
missing = df_final.isnull().sum()
missing_top = missing[missing > 0].sort_values(ascending=False).head(10)
if len(missing_top) > 0:
    for col, count in missing_top.items():
        pct = count / len(df_final) * 100
        print(f"   {col}: {count:,}개 ({pct:.1f}%)")
else:
    print("   ✅ 결측치 없음")



📊 Step 9: 최종 검증

1️⃣ 타겟 분포:
   정상 (Target=0): 15,616개 (79.66%)
   부도 (Target=1): 3,987개 (20.34%)

2️⃣ 데이터 샘플:


,bsns_year,stock_code,Company_Name,Status,F1_Equity_Growth,F1_Retained_Earnings_Ratio,F1_ROA,F1_Debt_Ratio,F1_Current_Ratio,F1_ROE,F1_Interest_Coverage,F2_KMV_DD,F3_Z_Score,F4_M_Score,M_Short_Term_Rate,M_Long_Term_Rate,M_Rate_Spread,M_Nominal_GDP_Growth,M_Real_GDP_Growth,M_Inflation,M_Exchange_Rate,Target,F1_Equity_Growth_change,F1_Equity_Growth_pct_change,F1_Equity_Growth_improving,F1_Retained_Earnings_Ratio_change,F1_Retained_Earnings_Ratio_pct_change,F1_Retained_Earnings_Ratio_improving,F1_ROA_change,F1_ROA_pct_change,F1_ROA_improving,F1_Debt_Ratio_change,F1_Debt_Ratio_pct_change,F1_Debt_Ratio_improving,F1_Current_Ratio_change,F1_Current_Ratio_pct_change,F1_Current_Ratio_improving,F1_ROE_change,F1_ROE_pct_change,F1_ROE_improving,F1_Interest_Coverage_change,F1_Interest_Coverage_pct_change,F1_Interest_Coverage_improving,target_year,corp_code,audit_prob,etc_prob,mda_prob,company_name,lex_sent_mean,lex_sent_sum,lex_pos_tf,lex_neg_tf,lex_pos_cnt,lex_neg_cnt,lex_abs_mean,lex_covered_tf
0,2023,00000010,신한은행,Distressed,NaN,0.046835,NaN,0.000000,NaN,NaN,0.346219,NaN,NaN,NaN,3.50,3.65,0.15,3.6,1.4,3.6,1305,1,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,2023,00149293,0.994615,0.165256,0.072178,신한은행,1.688633,1065.490638,357.333333,191.333333,109.333333,58.000000,6.420303,548.666667
1,2024,00000010,신한은행,Distressed,0.096433,0.047127,NaN,0.000000,NaN,NaN,0.390600,NaN,NaN,NaN,3.00,3.20,0.20,4.5,2.2,2.4,1360,1,0.000000,0.0,0.0,0.000292,0.624096,1.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.044381,12.818708,1.0,2024,00149293,0.994719,0.157525,0.049681,신한은행,1.843647,943.712022,363.000000,208.333333,108.333333,52.666667,6.379426,571.333333
2,2018,00000020,동화약품,Normal,NaN,0.656092,0.027167,0.197514,3.748255,0.033854,NaN,2.573247,4.470292,NaN,1.75,2.47,0.72,3.4,2.9,1.5,1100,0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.000000,0.0,2018,00119195,0.020391,0.136608,0.071009,동화약품,0.236295,425.137800,106.666667,47.000000,47.333333,16.333333,6.561111,153.666667
3,2019,00000020,동화약품,Normal,0.011681,0.656142,0.024982,0.199862,4.226125,0.031222,NaN,3.859740,4.249146,-2.555885,1.25,1.62,0.37,1.4,2.2,0.4,1166,0,0.000000,0.0,0.0,0.000051,0.007745,1.0,-0.002185,-8.044125,0.0,0.002348,1.188893,0.0,0.477870,12.749137,1.0,-0.002632,-7.774254,0.0,0.000000,0.000000,0.0,2019,00119195,0.014700,0.146697,0.202286,동화약품,0.562182,373.476605,112.000000,55.000000,51.000000,18.666667,6.355381,167.000000
4,2020,00000020,동화약품,Normal,0.138954,0.622960,0.066193,0.210060,3.300197,0.083795,NaN,1.196172,5.718731,-2.845481,0.50,1.39,0.89,0.4,-0.7,0.5,1180,0,0.127273,1000.0,1.0,-0.033183,-5.057250,0.0,0.041211,164.965904,1.0,0.010197,5.102202,0.0,-0.925928,-21.909630,0.0,0.052573,168.386363,1.0,0.000000,0.000000,0.0,2020,00119195,0.012528,0.121581,0.076270,동화약품,0.264443,303.944497,130.000000,74.333333,56.333333,26.666667,6.350085,204.333333



3️⃣ 결측치 요약 (상위 10개):
   F1_Interest_Coverage: 9,240개 (47.1%)
   F4_M_Score: 6,217개 (31.7%)
   F1_Equity_Growth: 3,057개 (15.6%)
   F3_Z_Score: 1,908개 (9.7%)
   F1_ROE: 835개 (4.3%)
   F1_ROA: 733개 (3.7%)
   F1_Retained_Earnings_Ratio: 383개 (2.0%)
   F2_KMV_DD: 382개 (1.9%)
   F1_Current_Ratio: 150개 (0.8%)
   F1_Debt_Ratio: 113개 (0.6%)


In [63]:
# ============================================
# Step 10: 최종 저장
# ============================================
print("\n" + "="*80)
print("💾 Step 10: 최종 저장")
print("="*80)
print()

OUTPUT_FILE = "최종_통합데이터_모델링용.csv"
df_final.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

file_size = Path(OUTPUT_FILE).stat().st_size / 1024 / 1024

print("✅ 저장 완료!")
print(f"   - 파일: {OUTPUT_FILE}")
print(f"   - 크기: {file_size:.2f} MB")
print(f"   - 샘플: {len(df_final):,}개")
print(f"   - 컬럼: {len(df_final.columns)}개")

print("\n" + "="*80)
print("🎉 최종 데이터 통합 완료!")
print("="*80)
print()
print("✅ 다음 단계: 모델 학습 (XGBoost, LightGBM 등)")



💾 Step 10: 최종 저장

✅ 저장 완료!
   - 파일: 최종_통합데이터_모델링용.csv
   - 크기: 12.55 MB
   - 샘플: 19,603개
   - 컬럼: 57개

🎉 최종 데이터 통합 완료!

✅ 다음 단계: 모델 학습 (XGBoost, LightGBM 등)


In [64]:
# ============================================
# Step 11: 컬럼 구조 확인
# ============================================
print("\n" + "="*80)
print("📋 Step 11: 컬럼 구조 확인")
print("="*80)
print()

print(f"총 컬럼 수: {len(df_final.columns)}개")
print()

# 컬럼 그룹별 정리
print("📊 컬럼 그룹:")

groups = {
    "키 컬럼": ['stock_code', 'target_year', 'company_name', 'Company_Name'],
    "타겟": ['Target', 'Status'],
    "KoBigBird": [c for c in df_final.columns if 'prob' in c],
    "감성사전": [c for c in df_final.columns if c.startswith('lex_')],
    "재무 피처": [c for c in df_final.columns if c.startswith('F1_') or c.startswith('F2_') or c.startswith('F3_') or c.startswith('F4_')],
    "추이 피처": [c for c in df_final.columns if '_change' in c or '_improving' in c],
    "거시경제": [c for c in df_final.columns if c.startswith('M_')]
}

for group_name, cols in groups.items():
    existing_cols = [c for c in cols if c in df_final.columns]
    if existing_cols:
        print(f"\n{group_name} ({len(existing_cols)}개):")
        for col in existing_cols[:5]:  # 처음 5개만 표시
            print(f"   - {col}")
        if len(existing_cols) > 5:
            print(f"   ... 외 {len(existing_cols)-5}개")



📋 Step 11: 컬럼 구조 확인

총 컬럼 수: 57개

📊 컬럼 그룹:

키 컬럼 (4개):
   - stock_code
   - target_year
   - company_name
   - Company_Name

타겟 (2개):
   - Target
   - Status

KoBigBird (3개):
   - audit_prob
   - etc_prob
   - mda_prob

감성사전 (8개):
   - lex_sent_mean
   - lex_sent_sum
   - lex_pos_tf
   - lex_neg_tf
   - lex_pos_cnt
   ... 외 3개

재무 피처 (31개):
   - F1_Equity_Growth
   - F1_Retained_Earnings_Ratio
   - F1_ROA
   - F1_Debt_Ratio
   - F1_Current_Ratio
   ... 외 26개

추이 피처 (21개):
   - F1_Equity_Growth_change
   - F1_Equity_Growth_pct_change
   - F1_Equity_Growth_improving
   - F1_Retained_Earnings_Ratio_change
   - F1_Retained_Earnings_Ratio_pct_change
   ... 외 16개

거시경제 (7개):
   - M_Short_Term_Rate
   - M_Long_Term_Rate
   - M_Rate_Spread
   - M_Nominal_GDP_Growth
   - M_Real_GDP_Growth
   ... 외 2개
